# World Model Training CliffWalking
Configs: 1-Standard, 2-MultiStep, 3-Delta

In [ ]:
import gymnasium as gym, numpy as np, torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
ENV_NAME = 'CliffWalking-v0'
torch.manual_seed(42)
env = gym.make(ENV_NAME)
obs, _ = env.reset()
STATE_DIM = 1
ACTION_DIM = env.action_space.n
env.close()
print(f'CliffWalking: {STATE_DIM}D state, {ACTION_DIM} actions')

In [ ]:
import torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
BATCH_SIZE, EPOCHS, LR, HIDDEN = 256, 200, 1e-3, 64
N_STEP, PATIENCE = 5, 8

In [ ]:
class WorldModel(nn.Module):
 def __init__(self, state_dim, action_dim, config=3, n_step=5, hidden=64):
  super().__init__()
  self.config, self.state_dim = config, state_dim
  in_dim = state_dim + (n_step * action_dim if config == 2 else action_dim)
  self.net = nn.Sequential(
   nn.Linear(in_dim, hidden), nn.ReLU(),
   nn.Linear(hidden, hidden), nn.ReLU(),
   nn.Linear(hidden, hidden), nn.ReLU(),
   nn.Linear(hidden, state_dim),
  )
 def forward(self, state, action):
  return self.net(torch.cat([state, action], dim=-1))
 def predict(self, state, action):
  out = self.forward(state, action)
  return state + out if self.config == 3 else out

In [ ]:
class EarlyStopping:
 def __init__(self, patience=8):
  self.patience, self.best, self.num_bad = patience, float('inf'), 0
 def step(self, loss):
  if loss < self.best - 1e-6:
   self.best, self.num_bad = loss, 0
   return False
  self.num_bad += 1
  return self.num_bad >= self.patience

In [ ]:
CKPT_DIR = Path('world_model/checkpoints')
CKPT_DIR.mkdir(exist_ok=True)
data_path = 'data/sequences_n5.npz'
try:
 data = np.load(data_path, allow_pickle=True)
 raw_data = {}
 for split in ['train', 'val', 'test']:
  raw_data[f'states_{split}'] = data[f'states_{split}'].astype(np.float32)
  raw_data[f'actions_onehot_{split}'] = data[f'actions_onehot_{split}'].astype(np.float32)
  raw_data[f'states_n_{split}'] = data[f'states_n_{split}'].astype(np.float32)
 print('Data loaded')
except FileNotFoundError:
 print('Data file not found')
 raw_data = None

In [ ]:
def build_dataset(config, data, n_step=5):
 datasets = {}
 for split in ['train', 'val', 'test']:
  if config == 2:
   X = np.concatenate([data[f'states_{split}'], data[f'actions_onehot_{split}'].reshape(len(data[f'actions_onehot_{split}']), -1)], axis=1)
   Y = data[f'states_n_{split}']
  else:
   X = np.concatenate([data[f'states_{split}'], data[f'actions_onehot_{split}'][:, 0, :]], axis=1)
   delta = (data[f'states_n_{split}'] - data[f'states_{split}']) / n_step
   Y = (data[f'states_{split}'] + delta if config == 1 else delta)
  datasets[split] = (torch.FloatTensor(X).to(device), torch.FloatTensor(Y).to(device))
 return datasets

In [ ]:
if raw_data:
 results = {}
 for cfg in [1, 2, 3]:
  print(f'\nTraining Config {cfg}')
  datasets = build_dataset(cfg, raw_data)
  model = WorldModel(STATE_DIM, ACTION_DIM, config=cfg, n_step=N_STEP).to(device)
  opt = optim.Adam(model.parameters(), lr=LR)
  crit = nn.MSELoss()
  es = EarlyStopping(PATIENCE)
  loader = DataLoader(TensorDataset(datasets['train'][0], datasets['train'][1]), batch_size=BATCH_SIZE, shuffle=True)
  for e in range(EPOCHS):
   model.train()
   loss_tr = 0
   for Xb, Yb in loader:
    opt.zero_grad()
    loss = crit(model(Xb[:, :STATE_DIM], Xb[:, STATE_DIM:]), Yb)
    loss.backward()
    opt.step()
    loss_tr += loss.item() * Xb.size(0)
   model.eval()
   with torch.no_grad():
    loss_v = crit(model(datasets['val'][0][:, :STATE_DIM], datasets['val'][0][:, STATE_DIM:]), datasets['val'][1]).item()
   if (e+1) % 30 == 0: print(f'  E{e+1}: {loss_tr/len(datasets["train"][0]):.6f} / {loss_v:.6f}')
   if es.step(loss_v): break
  with torch.no_grad():
   test_loss = crit(model(datasets['test'][0][:, :STATE_DIM], datasets['test'][0][:, STATE_DIM:]), datasets['test'][1]).item()
  name = ['Standard', 'MultiStep', 'Delta'][cfg-1]
  torch.save(model.state_dict(), CKPT_DIR / f'wm_CliffWalking_config{cfg}_{name}.pth')
  results[cfg] = test_loss
 print(f'\nResults:')
 for c in [1,2,3]:
  print(f'  Config {c}: {results.get(c, None)}')